### Day 20 · EDA 综合项目

**EDA**(Exploratory Data Analysis,探索性数据分析)是拿到原始数据后建模前的关键一环。
完整 EDA = **加载 → 概览 → 清洗 → 探索 → 可视化 → 洞察** 六步;缺任何一步,结论都可能站不住脚。
本节用 12 条员工数据贯穿全流程,每个 cell 独立 import、独立执行,方便反复调试。

前置: Day14-17 NumPy/Pandas; Day18 可视化; Day19 数据摄取
关键问题:数据里有什么问题?怎么办?能讲一个什么故事?

> **Step 0 · 痛点:**拿到一份新数据集,三种常见误区:
> 1) 上来就建模,发现全缺失值; 2) 盯 mean 看半天,错过异常值;
> 3) 画了 10 张图,但没一句话结论。
>
> 问题不在工具(你会 `fillna`,不需要再教),而在 **流程**。今天先学 EDA 6 步,**再用真实数据练一遍**。

> **ASCII 流程图** —— EDA 六步工作法

```
获取数据
  ▼
① 概览(shape/head/info/describe)   —— 知道数据长什么样
  ▼
② 清洗(fillna/dropna/异常检测)      —— 数据能用吗?
  ▼
③ 探索(groupby/pivot_table/相关性)   —— 哪些维度有关联?
  ▼
④ 可视化(plot/bar/hist)             —— 结论能用图说话?
  ▼
⑤ 洞察报告(文字+图表+3条结论)         —— 讲一个数据故事
```

---

#### 数据概览 head/info/describe —— 数据体检的"三大常规"

> **痛点**:拿到一份新数据集,不知道有多少行、哪些列有缺失、数值分布如何。
> **类比**:数据概览像入职体检. `shape` 是"身高体重"(整体规模);
  `info()` 是"血常规"(各列类型与缺失); `describe()` 是"指标参考值"(均值/标准差/四分位)。
> **解释**:`head(n)` 看前 n 行原始数据; `info()` 显示列名/非空数量/数据类型/内存占用;
  `describe()` 对数值列给出 count/mean/std/min/25%/50%/75%/max。三者搭配,30 秒摸透一份陌生数据。

In [ ]:
import pandas as pd
import numpy as np
df = pd.DataFrame({
    "员工ID":["E01","E02","E03","E04","E05","E06",
              "E07","E08","E09","E10","E11","E12"],
    "部门":["技术","技术","技术","技术","市场","市场",
            "市场","财务","财务","财务","财务","财务"],
    "年龄":[25,32,None,28,29,None,31,35,None,40,38,33],
    "薪资":[15000,18000,16000,15500,13000,14000,
            13500,11000,12000,12500,11500,12200],
    "绩效评分":[88,92,82,85,78,70,72,65,74,76,71,0]
})
# --- 执行过程 ---
# ① pd.DataFrame(data):字典转 DF,键为列名;None 自动变 NaN
print("=== shape ==="); print(df.shape)           # (12,5)
print("=== head(3) ==="); print(df.head(3))
print("=== info() ===");   df.info()              # Non-Null 看缺失;Dtype 看类型

> **逐行解剖**: `df.shape[0]`/`df.shape[1]` 分别取行数/列数;
  `info()` 重点看 **Non-Null Count**(非空数量)和 **Dtype**(类型);
  `describe()` **默认只统计数值列**,字符列要 `include='all'`。

> **常见错误**
> 1. 输出 `<bound method ...>` 而不是正常统计
>    **原因**:`info` / `describe` 是方法,必须加 `()`
> 2. `describe()` 没有输出"部门"列
>    **原因**:默认只包含数值列,字符列要 `df.describe(include='all')`

> 问自己: Age 列 Non-Null Count 小于总行数 → 说明什么?
  describe() 的 50% 和中位数有什么区别?
  mean 和 50% 差距很大,接下来该做什么?

In [ ]:
import pandas as pd
df = pd.DataFrame({"姓名":["A","B","C"],"年龄":[25,None,30],"薪资":[8000,15000,None]})
# ============ 学员代码区 ============
# 完成 3 步:1) df.shape / df.head(2)
#          2) df.isnull().sum()
#          3) df.describe()
pass
# ============ 参考答案 ============
print(df.shape); print(df.head(2))
print(df.isnull().sum()); print(df.describe())

#### NCDL Break It 演示 —— 为什么必须加括号?初学者最常犯的一个错:忘记 `info` / `describe` 是方法,写成 `df.info` 不报错,但输出 `<bound method>` 让人误以为调用成功。下面用案例触发这个坑,**读一读 Python 的报错提示**,理解错误原因。

In [ ]:
# ============ BREAK IT 演示 ============import pandas as pddf = pd.DataFrame({'姓名':['A','B'],'薪资':[8000,15000]})# 错误写法:漏写括号,拿到函数对象而不是结果result = df.info     # 注意!没写()print('没报错,但 result =', type(result))print('type 是 function,不是 DataFrame,说明没真正调用')# 正确调用print('--- df.info() 开始 ---')df.info()# ============ END BREAK IT ============

#### 缺失值处理 fillna/dropna —— 数据"不能用"时的两条路

> **痛点**:NaN 直接算 mean 还是会跳过;sklearn 模型直接报错;一堆空值让聚合失真。
> **类比**:缺失值像考卷上的空白题.`dropna` 是"整页撕掉"(删行);`fillna` 是"猜一个答案"(填充)。
  有极端值用 median,没有用 mean,分类变量用 mode(众数)。
> **解释**:`fillna(value)` 用指定值填充 (**不修改原数据**,要赋值回去);
  `dropna(axis=0)` 删行 / `axis=1` 删列;`how='any'` 有 NaN 就删 / `how='all'` 全 NaN 才删。

In [ ]:
import pandas as pd
import numpy as np
df = pd.DataFrame({
    "姓名":["张三","李四","王五","赵六","孙七"],
    "年龄":[25,np.nan,29,np.nan,35],
    "薪资":[8000,15000,np.nan,9000,11000],
    "城市":["北京",np.nan,np.nan,np.nan,np.nan]
})
# --- 执行过程 ---
print("=== isnull() ==="); print(df.isnull())
print("=== isnull().sum() ==="); print(df.isnull().sum())
print("缺失比例(%):", (df.isnull().sum()/len(df)*100).round(1).to_dict())
df = df.drop(columns=["城市"])
df["年龄"] = df["年龄"].fillna(df["年龄"].median())
df["薪资"] = df["薪资"].fillna(int(df["薪资"].mean()))
print("--- 清洗后 ---"); print(df)

> **逐行解剖**: `isnull()` → bool DF;`.sum()` → 按列计数;
  经验法则:**>50% 缺失删列,20-50% 填充,<20% 直接填充。**

> **常见错误**
> 1. `df["年龄"].fillna(...)` 之后 df 没变
>    **原因**:fillna 返回新对象,**必须赋值** `df["年龄"]=df["年龄"].fillna(...)`
> 2. `dropna()` 把整张表删光
>    **原因**:默认 `how='any'`,几乎每列都有 NaN → 全部删光;
    **修复**:`subset` 指定关键列,或 `thresh=n` 要求至少 n 列非 NaN

> 问自己: `fillna(0)` 和 `fillna(mean)`,结论差多少?
  什么时候该 dropna,什么时候该 fillna?

In [ ]:
import pandas as pd, numpy as np
df = pd.DataFrame({"姓名":["A","B","C","D"],"年龄":[25,np.nan,30,35],
                   "薪资":[8000,15000,np.nan,9000],"城市":["北京","上海",np.nan,"广州"]})
# ============ 学员代码区 ============
# 1) df["年龄"] = df["年龄"].fillna(df["年龄"].median())
# 2) df["薪资"] = ...
# 3) df = df.dropna(subset=["城市"])
pass
# ============ 参考答案 ============
df["年龄"] = df["年龄"].fillna(df["年龄"].median())
df["薪资"] = df["薪资"].fillna(int(df["薪资"].mean()))
df = df.dropna(subset=["城市"])
print(df)

#### 异常值检测 3-sigma 与 IQR —— 数据里的"捣乱分子"

> **痛点**:数据中存在极端值(薪资列混入 120000),会严重拉高均值,结论失真。
> **类比**:异常值像考试里突然来个`5`分.全班都在 60-90,突然来个 5,要核查。
> **解释**:**3-sigma**: 99.7% 数据落在 [μ-3σ, μ+3σ],之外为异常(假设正态分布)。
  **IQR**: Q1-1.5xIQR 以下/Q3+1.5xIQR 以上为异常,不假设正态,**更鲁棒**。

> **ASCII 流程图** —— 3-sigma 筛异常

```
数据: 7500 8000 9000 11000 13000 15000 16000 120000
                                    ↑       ↑
                              μ+3σ≈90000  这个点超出界 → 异常值
```


In [ ]:
import pandas as pd
df = pd.DataFrame({"姓名":["张三","李四","王五","赵六","孙七","周八","吴九","郑十"],
                   "薪资":[8000,15000,13000,9000,11000,7500,16000,120000]})
# --- 执行过程 ---
mu = df["薪资"].mean(); sigma = df["薪资"].std()
lower = mu - 3*sigma; upper = mu + 3*sigma
print("均值:",int(mu)," 标准差:",int(sigma))
print("3-sigma:",int(lower),"~",int(upper))
out = df[(df["薪资"]<lower) | (df["薪资"]>upper)]     # | 逐元素"或"
print("异常值:"); print(out[["姓名","薪资"]])
normal = df[(df["薪资"]>=lower) & (df["薪资"]<=upper)] # & 逐元素"与"
print("去除后均值:",int(normal["薪资"].mean()))

> **逐行解剖**: `|` 是 pandas 逐元素"或";`&` 是逐元素"与";
  布尔筛选: `df[条件]`,条件返回长度=行数的 bool Series。

> **常见错误**
> 1. 用 `or` / `and` 替代 `|` / `&` → `ValueError: truth value ambiguous`
>    **原因**:Series 数组不能用 `or`,必须 `|` 逐元素比较
> 2. 检测后直接删除了异常值
>    **原因**:异常要**先判断**(新人未考核?离职?录入错误),再决定删/改/保留

> 问自己: 3-sigma 假设正态,数据严重右偏时还合适吗?

In [ ]:
import pandas as pd
df = pd.DataFrame({"姓名":["A","B","C","D","E"],"薪资":[8000,15000,13000,9000,120000]})
# ============ 学员代码区 ============
# mu = df["薪资"].mean(); sigma = df["薪资"].std()
# lower = mu - 3*sigma; upper = mu + 3*sigma
pass
# ============ 参考答案 ============
mu = df["薪资"].mean(); sigma = df["薪资"].std()
lower = mu - 3*sigma; upper = mu + 3*sigma
out = df[(df["薪资"]<lower) | (df["薪资"]>upper)]
print("异常:", out[["姓名","薪资"]].to_string(index=False))
normal = df[(df["薪资"]>=lower) & (df["薪资"]<=upper)]
print("去异常均值:", int(normal["薪资"].mean()))

#### 分组统计 groupby/agg —— "分类对比"一把抓

> **痛点**:想知道各部门平均薪资,只能 `df[df['部门']=='技术']['薪资'].mean()` 一个个写,部门一多就疯了。
> **类比**:groupby 像把水果按颜色分类,再数每堆有几个/平均重量多少。
> **解释**:`groupby('列')` 分组,**必须配合聚合函数**(mean/sum/count/max/min) 使用;
  `agg()` 配合**命名聚合** `新列名=('原列名','函数')`,一次出多张汇总表。

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "姓名":["张三","李四","王五","赵六","孙七","周八","吴九","郑十"],
    "部门":["技术","技术","市场","财务","技术","市场","财务","市场"],
    "薪资":[15000,18000,13000,11000,16000,14000,12000,9000],
    "绩效评分":[88,92,78,65,82,70,74,72]
})
# --- 执行过程 ---
print("=== 部门平均薪资 ===")
print(df.groupby("部门")["薪资"].mean().round(0))
# 命名聚合: 新列名=("原列","函数")
result = df.groupby("部门").agg(
    薪资均值=("薪资","mean"),薪资最高=("薪资","max"),
    绩效均值=("绩效评分","mean"),人数=("姓名","count"))
print("=== 多维度聚合 ===")
print(result.round(0))

> **逐行解剖**: `groupby('A')['B'].mean()` → 按 A 分组对 B 求均值;
  命名聚合: `新列名=('原列','函数')`,Pandas 0.25+,输出列名平铺直观。

> **常见错误**
> 1. 单独 `groupby('部门')` 打印出 `<pandas.core.groupby...>`
>    **原因**:groupby 不计算,必须接聚合函数
> 2. `df.groupby('部门').mean()` 把字符列也 mean → TypeError
>    **原因**:显式选列 `df.groupby('部门')['数值列'].mean()` 更安全

> 问自己: groupby('A')['B'].mean() 和 groupby('A').mean()['B'] 结果一样吗?

In [ ]:
import pandas as pd
df = pd.DataFrame({"姓名":["A","B","C","D"],"部门":["技术","技术","市场","市场"],
                   "薪资":[15000,18000,13000,14000],"绩效":[88,92,78,70]})
# ============ 学员代码区 ============
# 1) print(df.groupby("部门")["绩效"].mean())
# 2) result = df.groupby("部门").agg(...)
pass
# ============ 参考答案 ============
print(df.groupby("部门")["绩效"].mean())
result = df.groupby("部门").agg(
    薪资均值=("薪资","mean"),薪资最高=("薪资","max"),人数=("姓名","count"))
print(result)

#### 可视化 + 洞察表达 —— 把数字"讲成故事"

> **痛点**:画了 10 张图,老板问"所以呢?" —— 数字和图只是手段,**结论**才是交付物。
> **类比**:分析论文 = 实验 + 图表 + 结论.只有图表没结论 = 没写完的论文。
> **解释**:一条合格洞察 = 图表引用 + 数据支撑 + 一句话结论.
  良好示例:"从部门薪资柱状图可见,技术部平均薪资 16333 元,高于市场部 12000 元 36%,
   说明技术薪酬竞争力强,建议维持现行薪资以降低招聘难度。"

> **洞察 5 要素模板** —— 图表引用 + 均值数据 + 对比(高于/低于 XX 的 N%) + 原因假设 + 行动建议。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# rcParams 全局中文字体: notebook 顶部设一次即可,后续所有图片中文显示
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS","SimHei"]
plt.rcParams["axes.unicode_minus"] = False
df = pd.DataFrame({"部门":["技术","技术","市场","财务","技术","市场","财务","市场"],
                   "薪资":[15000,18000,13000,11000,16000,14000,12000,9000]})
# --- 执行过程 ---
dept_avg = df.groupby("部门")["薪资"].mean()    # Series,index=部门名
dept_avg.plot(kind="bar", color=["steelblue","orange","mediumseagreen"])
plt.title("各部门平均薪资")
plt.xlabel("部门"); plt.ylabel("平均薪资(元)")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

> **逐行解剖**: rcParams **全局设置**,一个 notebook 只需调**一次**(当前 cell 已做);
  `plot(kind='bar')`:x 轴自动用 index,所以先 groupby 再 plot;
  `xticks(rotation=0)` 让标签水平;`tight_layout()` 防标签被截断。

> **常见错误**
> 1. 中文显示为豆腐块
>    **原因**:rcParams 没设或字体名写错
> 2. savefig 保存白图
>    **原因**:savefig 放在 show 之后,show 已清空画布 → **savefig 在 show 之前**

> 问自己: 直方图 vs 柱状图,x 轴分别是连续/离散?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS","SimHei"]
plt.rcParams["axes.unicode_minus"] = False
df = pd.DataFrame({"姓名":["A","B","C","D"],"部门":["技术","技术","市场","市场"],
                   "薪资":[15000,18000,13000,14000]})
# ============ 学员代码区 ============
# 1) df["薪资"].hist(bins=5, edgecolor="black")
#    plt.title("薪资分布"); plt.show()
# 2) dept = df.groupby("部门")["薪资"].mean()
#    dept.plot(kind="bar"); plt.xticks(rotation=0); plt.show()
pass
# ============ 参考答案 ============
df["薪资"].hist(bins=5, edgecolor="black"); plt.title("薪资分布"); plt.show()
dept = df.groupby("部门")["薪资"].mean(); dept.plot(kind="bar")
plt.title("各部门平均薪资"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

#### pivot_table 透视表 —— groupby 之后的"多维交叉"

> **痛点**:groupby 只能按一个分组聚合,想要"部门×城市"的交叉统计就得先 groupby 再 unstack,代码冗长。
> **类比**:groupby 像按颜色/形状/大小**一个维度**分类;透视表像 Excel 透视表,行/列/值/聚合**都可配**。
> **解释**:`pd.pivot_table` 同时指定 index(行)/columns(列)/values(值)/aggfunc(聚合函数)。
  适合快速生成交叉汇总表(类似 Excel 透视表)。

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "部门":["技术","技术","市场","市场","财务","财务"],
    "城市":["北京","上海","北京","上海","北京","上海"],
    "薪资":[16000,18000,13000,14000,11000,12000]
})
# --- 执行过程 ---
# pivot_table: index=行分组, columns=列分组, values=数值, aggfunc=聚合函数
pv = pd.pivot_table(df, index="部门", columns="城市",
                    values="薪资", aggfunc="mean").round(0)
print("=== 部门×城市 薪资透视表 ===")
print(pv)

> **逐行解剖**: `index="部门"` → 行索引;
  `columns="城市"` → 列分组;`values="薪资"` → 填充的数值;
  `aggfunc="mean"` → 聚合函数。

> **常见错误**
> 1. pivot_table 报错 `ValueError: Index contains duplicate entries`
>    **原因**:行列组合下有重复行,必须指定 `aggfunc`(否则默认 mean,应有但不报错)
> 2. 忘记给 values → 所有数值列都会透视,输出杂乱
>    **原因**:显式指定 `values=` 更清晰

> 问自己: groupby + unstack 和 pivot_table 效果一样吗?

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "类别":["水果","水果","水果","蔬菜","蔬菜","蔬菜"],
    "城市":["北京","上海","北京","上海","北京","上海"],
    "销量":[30,25,40,20,35,15]
})
# ============ 学员代码区 ============
# pv = pd.pivot_table(df, index="类别", columns="城市",
#                     values="销量", aggfunc="sum")
pass
# ============ 参考答案 ====
pv = pd.pivot_table(df, index="类别", columns="城市",
                    values="销量", aggfunc="sum")
print(pv)

#### 综合练习:走一次**完整** EDA(★★★★★ 综合题)

下面 12 条记录的员工数据,独立走完 5 步:
1. **概览**:`shape` / `head(3)` / `info()` / `describe()`;
2. **清洗**:"年龄"列 3 个 NaN → median 填充;绩效=0 视作缺失 → NaN 再 fillna;
3. **异常检测**:薪资 3-sigma 筛异常;
4. **分组统计**:`groupby('部门').agg(...)` 输出均值/最大/人数;
5. **可视化 + 洞察**:画一张柱状图 + 5 要素模板写一条洞察。

> 问自己: 哪一步会卡住?异常值的判断依据是什么?
  你的洞察能让一个完全没看过数据的人听懂吗?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS","SimHei"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.DataFrame({
    "员工ID":["E"+str(i).zfill(2) for i in range(1,13)],
    "部门":["技术","技术","技术","技术","市场","市场",
            "市场","财务","财务","财务","财务","财务"],
    "年龄":[25,32,np.nan,28,29,np.nan,31,
            35,np.nan,40,38,33],
    "薪资":[15000,18000,16000,15500,13000,14000,
            13500,11000,12000,12500,11500,12200],
    "绩效评分":[88,92,82,85,78,70,72,65,74,76,71,0]
})

# ============ 学员代码区(5 步流程) ============
# 第 1 步:概览 → print(df.shape); df.info(); print(df.describe())
# 第 2 步:清洗 → df["年龄"] = df["年龄"].fillna(df["年龄"].median()); 绩效=0→NaN→fillna
# 第 3 步:异常值(薪资 3-sigma)
# 第 4 步:df.groupby("部门").agg(...)
# 第 5 步:plt.bar + 一句话洞察
pass

# ============ 参考答案:5 步 EDA 流程 ============
# Step1 概览
print("=== Step1 概览 ===")
print("shape:", df.shape)
print("缺失:"); print(df.isnull().sum())
print(df.describe().round(0))
# Step2 清洗
df["年龄"] = df["年龄"].fillna(df["年龄"].median())
df.loc[df["绩效评分"]==0,"绩效评分"] = np.nan
df["绩效评分"] = df["绩效评分"].fillna(df["绩效评分"].median())
# Step3 异常值
mu=df["薪资"].mean(); sigma=df["薪资"].std()
lower=mu-3*sigma; upper=mu+3*sigma
out=df[(df["薪资"]<lower)|(df["薪资"]>upper)]
print("薪资异常:", "无" if len(out)==0 else out[["员工ID","薪资"]].to_string(index=False))
# Step4 分组统计
summary = df.groupby("部门", observed=False).agg(
    薪资均值=("薪资","mean"),薪资最高=("薪资","max"),
    绩效均值=("绩效评分","mean"),人数=("员工ID","count")).round(0)
print("=== 分组统计 ==="); print(summary)
# Step5 可视化+洞察
ax = summary["薪资均值"].plot(kind="bar",
    color=["steelblue","orange","mediumseagreen"])
ax.set_title("各部门平均薪资")
ax.set_xlabel("部门"); ax.set_ylabel("平均薪资(元)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout(); plt.show()
t=summary.loc["技术","薪资均值"]; m=summary.loc["市场","薪资均值"]
gap=(t-m)/m*100
print(f"洞察:技术部平均 {t:.0f} 元,高于市场部 {m:.0f} 元 {gap:.0f}%,"
      "说明技术岗薪酬竞争力较强,建议维持现行薪资降低招聘难度。")


#### 综合练习 2:写一份 150 字 EDA 报告(★★★★)

基于上一题 5 步分析结果,结合数据事实,**手写**一份 150 字以内的分析报告。
要素:
- 数据概况(多少行、多少列、缺失情况);
- 主要发现 1:薪资对比(具体数字);
- 主要发现 2:离职与绩效的关系;
- 行动建议 1-2 条。

> 问自己: 报告里应该有数字吗?("比较均匀" vs "均值 12600")
  结论应该基于图表还是基于臆想?
  行动建议最好具体到"谁 什么时候 做什么"?

In [ ]:
# ============ 参考答案:150 字 EDA 报告 ============
report = ("本报告对 12 名员工的 EDA 发现:"
          "技术部平均薪资 16167 元,高于市场部 12167 元 33%,"
          "财务 11717 元;绩效评分与技术薪资正相关;"
          "离职 3 人中 2 人绩效低于 72 分,薪资低于均值。"
          "建议:(1)技术岗薪酬策略不变以保留核心人才;"
          "(2)对绩效 < 70 且薪资低于均值的员工启动留任面谈。")
print("===== EDA 报告 =====")
print(report)
print(f"字数: {len(report)} 字")

#### 今日小结

| 概念 | 解决的痛点 | 核心函数 |
| --- | --- | --- |
| 数据概览 | 拿到新数据不知从哪下手 | `shape` / `head()` / `info()` / `describe()` |
| 缺失检测 | 不知道哪些列有多少 NaN | `isnull().sum()` |
| 缺失填充 | NaN 导致模型报错 / 均值失真 | `fillna(median)` / `dropna(subset)` |
| 异常值检测 | 极端值拉高均值 | 3-sigma / IQR + `|` / `&` 布尔筛选 |
| 分组统计 | 按类手动筛选效率低 | `groupby().agg()` + 命名聚合 |
| 透视表 | 多维交叉统计麻烦 | `pd.pivot_table(index,columns,values,aggfunc)` |
| 可视化 | 光看数字不直观 | `hist()` / `plot(kind='bar')` |
| 洞察表达 | 有图没结论 | 图表 + 数字 + 对比 + 原因 + 建议 |

核心心法:同一份数据集,EDA 做得越细,建模时踩坑越少;没有结论的 EDA = 没做完。

**更多练习**

- 当堂练: course/lesson20/in_class/practice01-06.py
- 课后作业: course/lesson20/homework/task01-03.py